# Enhanced Comparison: Multi-PC Distance + Variable Top-K

Two improvements over the basic PCA vs DIIM comparison:
1. **Dynamic PCA sector ranking** using Euclidean distance in multi-PC space (1, 2, 3, 5 PCs)
2. **Variable top-k sweep** (k = 3, 5, 7, 10, 15, 20) to see how each method scales

**Methods compared:**
- PCA (PC1 only), PCA (2-PC), PCA (3-PC), PCA (5-PC)
- DIIM (gold standard, requires simulation)
- Sensitivity (perturb each sector by 10%, rank by marginal loss reduction)
- Network Centrality (composite of eigenvector, PageRank, betweenness)

**Note:** Runtime ~10-15 minutes.

## 1. Load Libraries & Functions

In [ ]:
setwd("..")

library(openxlsx)
library(igraph)
library(ggplot2)
library(reshape2)
library(dplyr)

source_notebook <- function(nb_path) {
  nb <- jsonlite::fromJSON(nb_path, simplifyVector = FALSE)
  code_cells <- Filter(function(cell) cell$cell_type == "code", nb$cells)
  code_lines <- unlist(lapply(code_cells, function(cell) {
    paste(unlist(cell$source), collapse = "")
  }))
  all_code <- paste(code_lines, collapse = "\n")
  eval(parse(text = all_code), envir = globalenv())
}

source_notebook("functions.ipynb")

results_dir <- "results"
dir.create(results_dir, showWarnings = FALSE, recursive = TRUE)

## 2. DIIM Sector Ranking

Run a full DIIM simulation and rank sectors by their peak economic loss. This is the gold standard but requires simulation.

In [ ]:
diim_rank_sectors <- function(q0, A_star, c_star, x,
                              lockdown_duration, total_duration,
                              days_in_year = 366) {
    model <- DIIM(q0, A_star, c_star, x, lockdown_duration, total_duration,
                  days_in_year = days_in_year)
    max_el <- apply(model$EL_evolution, 1, max)
    ranked_sectors <- order(max_el, decreasing = TRUE)
    return(list(ranked_sectors = ranked_sectors, max_el = max_el))
}

## 3. Sensitivity Sector Ranking

Perturb each sector's q0 by -10%, run DIIM, and rank by marginal loss reduction. Expensive (n simulations) but doesn't require arbitrary choices like PCA.

In [ ]:
sensitivity_rank_sectors <- function(q0, A_star, c_star, x,
                                     lockdown_duration, total_duration,
                                     days_in_year = 366) {
    n <- length(q0)
    base_model <- DIIM(q0, A_star, c_star, x, lockdown_duration, total_duration,
                       days_in_year = days_in_year)
    base_loss <- base_model$total_economic_loss

    marginal_reductions <- numeric(n)
    for (i in 1:n) {
        q0_test    <- q0
        q0_test[i] <- q0_test[i] * 0.9
        model_test <- DIIM(q0_test, A_star, c_star, x, lockdown_duration,
                           total_duration, days_in_year = days_in_year)
        marginal_reductions[i] <- base_loss - model_test$total_economic_loss
    }

    ranked_sectors <- order(marginal_reductions, decreasing = TRUE)
    return(list(ranked_sectors = ranked_sectors,
               marginal_reductions = marginal_reductions))
}

## 4. Top-K Evaluation Helper

Given a sector ranking, intervene on the top k sectors and compute the % loss reduction vs baseline.

In [ ]:
evaluate_topk <- function(ranked_sectors, k, q0, A_star, c_star, x,
                          lockdown_duration, total_duration,
                          base_loss, days_in_year = 366) {
    key_sectors <- ranked_sectors[1:k]
    model_int <- DIIM(q0, A_star, c_star, x, lockdown_duration, total_duration,
                      key_sectors = key_sectors, days_in_year = days_in_year)
    reduction <- base_loss - model_int$total_economic_loss
    pct_reduction <- reduction / base_loss * 100
    return(list(k = k, sectors = key_sectors,
               loss = model_int$total_economic_loss,
               reduction = reduction, pct_reduction = pct_reduction))
}

## 5. Run Scenario Function

For a given scenario:
1. Run DIIM baseline (no intervention).
2. Compute rankings from all methods: PCA (1/2/3/5 PCs), DIIM, Sensitivity, Network Centrality.
3. For each method and each k, compute the % loss reduction.
4. Return a results table.

In [ ]:
run_scenario <- function(scenario_name, data_loader,
                         lockdown_duration = 55, total_duration = 751,
                         days_in_year = 366,
                         k_values = c(3, 5, 7, 10, 15, 20)) {
    cat(sprintf("\n%s Scenario\n", scenario_name))

    data   <- data_loader()
    A      <- data$A
    x      <- data$x
    q0     <- data$q0
    q0[q0 == 0] <- 1e-8
    c_star <- data$c_star
    A_star <- data$A_star
    n      <- length(q0)

    k_values <- k_values[k_values < n]
    cat(sprintf("  Sectors: %d, k values: %s\n", n, paste(k_values, collapse = ", ")))

    # baseline (no intervention)
    base_model <- DIIM(q0, A_star, c_star, x, lockdown_duration, total_duration,
                       days_in_year = days_in_year)
    base_loss <- base_model$total_economic_loss
    cat(sprintf("  Baseline loss: %.2f\n\n", base_loss))

    # --- Step 1: Compute all rankings ---

    # PCA variants (1, 2, 3, 5 PCs)
    cat("  Computing PCA rankings...\n")
    pca_variants <- list()
    pc_counts <- c(1, 2, 3, 5)
    pc_counts <- pc_counts[pc_counts <= n]
    for (npc in pc_counts) {
        pca_res <- pca_rank_sectors(A, n_pcs = npc)
        label <- if (npc == 1) "PCA (PC1 only)" else sprintf("PCA (%d-PC distance)", npc)
        pca_variants[[label]] <- pca_res
        cat(sprintf("    %s: cumulative variance = %.1f%%, top-5 = [%s]\n",
            label, pca_res$cumulative_var * 100,
            paste(pca_res$ranked_sectors[1:min(5, n)], collapse = ", ")))
    }

    # DIIM
    cat("  Computing DIIM ranking...\n")
    diim_res <- diim_rank_sectors(q0, A_star, c_star, x,
                                  lockdown_duration, total_duration, days_in_year)
    cat(sprintf("    DIIM top-5: [%s]\n",
        paste(diim_res$ranked_sectors[1:5], collapse = ", ")))

    # Sensitivity
    cat("  Computing Sensitivity ranking...\n")
    sens_res <- sensitivity_rank_sectors(q0, A_star, c_star, x,
                                        lockdown_duration, total_duration, days_in_year)
    cat(sprintf("    Sensitivity top-5: [%s]\n",
        paste(sens_res$ranked_sectors[1:5], collapse = ", ")))

    # Network Centrality (from functions.ipynb)
    cat("  Computing Network Centrality ranking...\n")
    cent_res <- centrality_rank_sectors(A_star)
    cat(sprintf("    Centrality top-5: [%s]\n",
        paste(cent_res$ranked_sectors[1:5], collapse = ", ")))

    # --- Step 2: Evaluate all methods at each k ---
    cat("\n  Evaluating all methods at each k...\n")

    all_methods <- list()
    for (label in names(pca_variants)) {
        all_methods[[label]] <- pca_variants[[label]]$ranked_sectors
    }
    all_methods[["DIIM"]]               <- diim_res$ranked_sectors
    all_methods[["Sensitivity"]]         <- sens_res$ranked_sectors
    all_methods[["Network Centrality"]]  <- cent_res$ranked_sectors

    results <- data.frame(
        scenario = character(), method = character(),
        k = integer(), pct_reduction = numeric(),
        sectors = character(), stringsAsFactors = FALSE)

    for (mname in names(all_methods)) {
        for (k in k_values) {
            ev <- evaluate_topk(all_methods[[mname]], k, q0, A_star, c_star, x,
                                lockdown_duration, total_duration, base_loss, days_in_year)
            results <- rbind(results, data.frame(
                scenario = scenario_name, method = mname, k = k,
                pct_reduction = ev$pct_reduction,
                sectors = paste(ev$sectors, collapse = ","),
                stringsAsFactors = FALSE))
        }
    }

    # print summary table
    cat("\n  Results (% loss reduction):\n")
    summary_table <- reshape2::dcast(results, method ~ k, value.var = "pct_reduction")
    print(summary_table)

    return(results)
}

## 6. Run Both Scenarios & Save

Run COVID-19 (15 sectors) and Manpower (107 sectors) scenarios, then combine results into one CSV.

In [ ]:
# COVID-19 (15 sectors)
covid_results <- run_scenario("COVID-19", download_data,
    lockdown_duration = 55, total_duration = 751,
    days_in_year = 366, k_values = c(3, 5, 7, 10, 12))

# Manpower (107 sectors)
manpower_results <- run_scenario("Manpower", download_manpower_data,
    lockdown_duration = 55, total_duration = 751,
    days_in_year = 365, k_values = c(3, 5, 7, 10, 15, 20, 30))

combined <- rbind(covid_results, manpower_results)
write.csv(combined, file.path(results_dir, "enhanced_comparison.csv"), row.names = FALSE)
cat("\nSaved enhanced_comparison.csv\n")

## 7. Plot — Top-K Comparison (per scenario)

Line chart showing % loss reduction vs k for each method. PCA variants are solid lines; DIIM and sensitivity are dashed.

In [ ]:
plot_topk_comparison <- function(data, scenario_name, filename) {
    plot_data <- data[data$scenario == scenario_name, ]
    plot_data$method_type <- ifelse(grepl("PCA", plot_data$method), "PCA Variant",
        ifelse(plot_data$method == "DIIM", "DIIM",
            ifelse(plot_data$method == "Sensitivity", "Sensitivity", "Other")))

    p <- ggplot(plot_data, aes(x = k, y = pct_reduction,
                               color = method, linetype = method_type)) +
        geom_line(linewidth = 1.1) +
        geom_point(size = 2.5) +
        scale_color_brewer(palette = "Set2", name = "Method") +
        scale_linetype_manual(
            values = c("PCA Variant" = "solid", "DIIM" = "dashed",
                       "Sensitivity" = "dotdash", "Other" = "dotted"),
            name = "Type") +
        labs(title = sprintf("%s: Loss Reduction vs k", scenario_name),
             subtitle = "Comparing PCA variants, DIIM, Sensitivity, and Network Centrality",
             x = "Number of Sectors Intervened (k)",
             y = "% Loss Reduction") +
        theme_minimal() +
        theme(plot.title = element_text(face = "bold", size = 14),
              legend.position = "bottom", legend.box = "vertical") +
        guides(color = guide_legend(nrow = 3), linetype = guide_legend(nrow = 1))

    ggsave(file.path(results_dir, filename), p, width = 12, height = 7)
    cat(sprintf("Saved %s\n", filename))
}

plot_topk_comparison(combined, "COVID-19", "enhanced_covid_topk.png")
plot_topk_comparison(combined, "Manpower", "enhanced_manpower_topk.png")

## 8. Plot — PCA Variant Comparison

Side-by-side faceted plot showing whether using more PCs (2, 3, 5) improves PCA's ranking over PC1-only.

In [ ]:
pca_data <- combined[grepl("PCA", combined$method), ]
if (nrow(pca_data) > 0) {
    p_pca <- ggplot(pca_data, aes(x = k, y = pct_reduction,
                                   color = method, shape = method)) +
        geom_line(linewidth = 1.2) +
        geom_point(size = 3) +
        facet_wrap(~scenario, scales = "free_y") +
        scale_color_brewer(palette = "Dark2", name = "PCA Variant") +
        labs(title = "PCA Variant Comparison: PC1 vs Multi-PC Distance",
             x = "Top-k Sectors", y = "% Loss Reduction") +
        theme_minimal() +
        theme(plot.title = element_text(face = "bold", size = 14),
              legend.position = "bottom")
    ggsave(file.path(results_dir, "enhanced_pca_variants.png"), p_pca,
           width = 12, height = 6)
    cat("Saved enhanced_pca_variants.png\n")
}

## 9. Plot — Best Method at Each k

In [ ]:
winners <- combined %>%
    group_by(scenario, k) %>%
    filter(pct_reduction == max(pct_reduction)) %>%
    ungroup()

p_winner <- ggplot(winners, aes(x = factor(k), y = pct_reduction, fill = method)) +
    geom_bar(stat = "identity", position = "dodge", alpha = 0.85) +
    facet_wrap(~scenario, scales = "free_y") +
    scale_fill_brewer(palette = "Set2", name = "Best Method") +
    labs(title = "Best Method at Each k",
         x = "Number of Sectors (k)", y = "% Loss Reduction") +
    theme_minimal() +
    theme(plot.title = element_text(face = "bold", size = 14),
          legend.position = "bottom")

ggsave(file.path(results_dir, "enhanced_best_at_k.png"), p_winner,
       width = 12, height = 6)
cat("Saved enhanced_best_at_k.png\n")

## 10. Summary

Print which method wins at each k, and show whether multi-PC PCA improves over PC1-only.

In [ ]:
cat("ENHANCED COMPARISON SUMMARY\n\n")

for (sc in unique(combined$scenario)) {
    cat(sprintf("--- %s ---\n", sc))
    sc_data <- combined[combined$scenario == sc, ]

    for (k_val in sort(unique(sc_data$k))) {
        k_data <- sc_data[sc_data$k == k_val, ]
        best   <- k_data[which.max(k_data$pct_reduction), ]
        cat(sprintf("  k=%2d: Best = %-25s (%.2f%%)", k_val, best$method, best$pct_reduction))

        # also show best PCA variant for comparison
        pca_vals <- k_data[grepl("PCA", k_data$method), ]
        if (nrow(pca_vals) > 0) {
            best_pca <- pca_vals[which.max(pca_vals$pct_reduction), ]
            cat(sprintf("  |  Best PCA = %-20s (%.2f%%)", best_pca$method, best_pca$pct_reduction))
        }
        cat("\n")
    }
    cat("\n")
}

# multi-PC improvement at k=5
cat("PCA Improvement from Multi-PC Distance (at k=5):\n")
for (sc in unique(combined$scenario)) {
    sc_data  <- combined[combined$scenario == sc & combined$k == 5, ]
    pc1_only <- sc_data[sc_data$method == "PCA (PC1 only)", "pct_reduction"]
    multi_pc <- sc_data[grepl("2-PC|3-PC|5-PC", sc_data$method), ]
    if (length(pc1_only) > 0 && nrow(multi_pc) > 0) {
        best_multi  <- multi_pc[which.max(multi_pc$pct_reduction), ]
        improvement <- best_multi$pct_reduction - pc1_only
        cat(sprintf("  %s: PC1=%.3f%% -> %s=%.3f%% (delta=%.3f%%)\n",
            sc, pc1_only, best_multi$method, best_multi$pct_reduction, improvement))
    }
}

cat("\nDone!\n")